# 07. Advanced Analysis — 고급 분석

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/07_advanced.ipynb)

K-IFRS 주석(Notes) 통합 접근, 유형자산 변동표, 관계기업 분석, 거버넌스 분석 등 심화 모듈을 다룬다.

- K-IFRS 주석 통합 접근 (12개 항목)
- 미등록 키워드 직접 조회 (23개)
- 유형자산 변동표
- 관계기업·공동기업
- 이사회와 감사제도
- 거버넌스·리스크 분석
- Result 객체와 전체 일괄 조회

📖 [문서 보기](https://eddmpython.github.io/dartlab/docs/tutorials/advanced)

In [ ]:
!pip install -q dartlab

In [ ]:
import dartlab

samsung = dartlab.Company("005930")

## K-IFRS 주석 — Notes 통합 접근

`c.notes`로 12개 K-IFRS 주석 항목에 통합 접근한다. 영문 속성과 한글 키 모두 지원한다.

In [ ]:
samsung.notes.inventory

In [ ]:
samsung.notes.receivables

In [ ]:
samsung.notes["차입금"]

In [ ]:
print("영문 키:", samsung.notes.keys())
print("한글 키:", samsung.notes.keys_kr())

## 미등록 키워드 직접 조회 (23개)

`get("notesDetail", keyword=...)`로 직접 호출한다.

지원 키워드: 재고자산, 주당이익, 충당부채, 차입금, 매출채권, 리스, 투자부동산, 무형자산, 법인세, 특수관계자, 약정사항, 금융자산, 공정가치, 이익잉여금, 금융부채, 기타포괄손익, 사채, 종업원급여, 퇴직급여, 확정급여, 재무위험, 우발부채, 담보

In [ ]:
inventory = samsung.get("notesDetail", keyword="재고자산")

print(f"키워드: {inventory.keyword}")
print(f"분석 연도: {inventory.nYears}")

In [ ]:
inventory.tableDf

In [ ]:
for year, periods in inventory.tables.items():
    for p in periods:
        print(f"[{year}] {p.period} (패턴: {p.pattern})")
        for item in p.items[:3]:
            print(f"  {item.name}: {item.values}")
        print()

## 유형자산 변동표

In [ ]:
samsung.notes.tangibleAsset

In [ ]:
ta = samsung.get("tangibleAsset")
print(f"신뢰도: {ta.reliability}")
if ta.warnings:
    print(f"경고: {ta.warnings}")

In [ ]:
ta.movementDf

## 관계기업·공동기업

In [ ]:
samsung.notes.affiliates

In [ ]:
aff = samsung.get("affiliates")
if aff:
    for year, profiles in aff.profiles.items():
        for p in profiles[:3]:
            print(f"[{year}] {p.name}: {p.ratio}%, 장부가 {p.bookValue}")

## 이사회와 감사제도

In [ ]:
samsung.boardOfDirectors

In [ ]:
samsung.auditSystem

In [ ]:
samsung.internalControl

In [ ]:
board = samsung.get("boardOfDirectors")
if board:
    print(board.committeeDf)

## 거버넌스·리스크 종합

In [ ]:
samsung.relatedPartyTx

In [ ]:
samsung.contingentLiability

In [ ]:
samsung.sanction

In [ ]:
samsung.riskDerivative

In [ ]:
rpt = samsung.get("relatedPartyTx")
if rpt:
    print("=== 매출입 거래 ===")
    print(rpt.revenueTxDf)
    print("\n=== 채무보증 ===")
    print(rpt.guaranteeDf)

In [ ]:
cl = samsung.get("contingentLiability")
if cl:
    print(cl.lawsuitDf)

## Result 객체 접근

property는 대표 DataFrame 하나를 반환한다. 모듈의 전체 데이터가 필요하면 `get()`을 사용한다.

In [ ]:
samsung.audit

In [ ]:
result = samsung.get("audit")
result.feeDf

## 여러 종목 비교 분석

In [ ]:
import polars as pl

codes = ["005930", "000660", "035420"]
rows = []

for code in codes:
    c = dartlab.Company(code)
    bs = c.BS
    if bs is not None:
        cols = [col for col in bs.columns if col != "account"]
        last_year = cols[-1]
        row = bs.filter(pl.col("account") == "자산총계")
        if row.height > 0:
            rows.append({
                "기업": c.corpName,
                "자산총계": row[last_year][0],
            })

pl.DataFrame(rows)

## 전체 일괄 조회

In [ ]:
d = samsung.all()
print(list(d.keys()))

In [ ]:
d["notes"]["inventory"]

---

다음: [8. 기업 간 비교](./08_cross_company.ipynb) — 섹터 분류, 시장 순위